# OOS validation — random hollowing per signal

## The problem this notebook solves

The 12 signals in `best_signals_aggregate.ipynb` were selected AFTER seeing their full-history performance. Even if each individual signal is legit, the SELECTION ITSELF is a form of forward-looking bias — we picked winners from ~800 candidates, and by chance some noise-fitters will pass any gate.

## The test

For each candidate signal in a large pool:
1. Generate a **random 20% OOS mask, DIFFERENT per signal** (contiguous blocks of ~20 days to preserve autocorrelation)
2. **Sign, gate pass/fail, and metrics are computed ON THE 80% IS BARS ONLY** — the OOS bars are truly unseen at selection time
3. Selected signals ("IS-passers") are those clearing the gate on their own IS bars
4. **Aggregated OOS PnL**: at each date D, average the PnL across signals where D is OOS for THAT signal

If the aggregated OOS PnL curves upward over time → selection captures genuine edge. If flat/negative → we're overfitting.

**Monte Carlo**: repeat over many random seeds. Report the distribution of OOS SR across seeds.

## Framework properties

- **Signal COMPUTATION** uses all bars (no gaps in rolling windows) — the mask only affects evaluation/selection
- **Signal timing (shift and PIT)** is unchanged from the aggregate: shift(2) × c2c returns; intraday winners use shift(1) × RD_ongap
- **Cost model** identical to `cta.Simulate` and `simulate_by_dollars`
- **Gates identical to aggregate § 3** (SR, β, pos_yr, max_dd, SR_of_SR, n_years) — but computed IS-only


In [ ]:
import importlib, sys, warnings, csv
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path

warnings.filterwarnings("ignore", message="findfont: .* not found")
_inst = {f.name for f in fm.fontManager.ttflist}
_cand = ["Hiragino Sans GB","Heiti TC","Songti SC","PingFang HK","PingFang SC","PingFang TC",
         "Arial Unicode MS","Noto Sans CJK TC","DejaVu Sans"]
mpl.rcParams["font.family"] = [f for f in _cand if f in _inst] or ["DejaVu Sans"]
mpl.rcParams["axes.unicode_minus"] = False

import cta
for _m in ["cta.asset","cta.operators","cta.simulate","cta.simulate_dollars",
          "cta.large_trader","cta.options","cta.three_majors","cta.tsmc_events",
          "cta.us_indexes","cta.signal_stats"]:
    if _m in sys.modules: importlib.reload(sys.modules[_m])
importlib.reload(cta)

EVAL_START, EVAL_END = "2013-01-01", "2026-07-27"

ASSET = cta.load_asset("mtx","1d"); cta.set_active_asset(ASSET)
close  = ASSET["close"].astype(float);  open_  = ASSET["open"].astype(float)
high   = ASSET["high"].astype(float);   low    = ASSET["low"].astype(float)
back_close = ASSET["back_close"].astype(float)
n_close = ASSET["night_close"].astype(float)
volume = ASSET["volume"].astype(float) if "volume" in ASSET.columns else None

# Return series
ret_c2c    = close.pct_change()
ret_ongap  = open_ / n_close.shift(1) - 1

# Costs (matching aggregate + intraday)
cost_c2c   = 20.0/(close * 50.0) + 0.00002
cost_ongap = 20.0/(n_close.shift(1) * 50.0) + 0.00002

# Trading calendar
CAL = ASSET.index[(ASSET.index >= EVAL_START) & (ASSET.index <= EVAL_END)]
print(f"eval window: {CAL[0].date()} → {CAL[-1].date()}   {len(CAL)} bars")


## 1. Build the full candidate POOL — 45 aggregate + 4 intraday variants

All variants that would have gone through the aggregate's gate. Each will get its own random OOS mask.


In [ ]:
def _bd(s): return np.tanh(s.replace([np.inf,-np.inf],np.nan))
def _dev(x,w): return x-cta.InstMean(w,x)
def _selfz(x,w): mu=cta.InstMean(w,x); sd=cta.InstStdev(w,x).replace(0,np.nan); return (x-mu)/sd
def _selfz_winsor(x,w,c=3.0): return _selfz(x,w).clip(-c,c)/c
def _robust_z(x,w):
    med=x.rolling(w,min_periods=max(3,w//2)).median()
    mad=(x-med).abs().rolling(w,min_periods=max(3,w//2)).median()
    return (x-med)/(1.4826*mad).replace(0,np.nan)
def _sign_thresh(x,w,t=0.5):
    z=_selfz(x,w); return pd.Series(np.where(z>t,1.0,np.where(z<-t,-1.0,0.0)),index=x.index)
def _rank_c(x,w): return (cta.InstRank(w,x)-0.5)*2
def _dm_tanh(s,W):
    dm=s-s.rolling(W,min_periods=20).mean(); sd=s.rolling(W,min_periods=20).std().replace(0,np.nan); return np.tanh(dm/sd)
def _chg_z_tanh(s,N,W=60):
    chg=s.diff(N); return np.tanh((chg-chg.rolling(W,min_periods=10).mean())/chg.rolling(W,min_periods=10).std().replace(0,np.nan))

# Raw features
tx_b10=cta.load_large_trader("TX","top10_buy").astype(float); tx_s10=cta.load_large_trader("TX","top10_sell").astype(float)
tx_oi=cta.load_large_trader("TX","total_oi").astype(float); tx_net10=cta.load_large_trader("TX","top10_net_pct").astype(float)
tx_net5=cta.load_large_trader("TX","top5_net_pct").astype(float)
tx_log_ratio=np.log(tx_b10.replace(0,np.nan)/tx_s10.replace(0,np.nan))
tx_conv=np.sign(tx_net10)*((tx_b10+tx_s10)/tx_oi).astype(float); tx_top5_lead=tx_net5-tx_net10
put_mo=cta.load_option_daily_total("oi","put","monthly").astype(float)
call_all=cta.load_option_daily_total("oi","call","all").astype(float)
put_all=cta.load_option_daily_total("oi","put","all").astype(float)
call_mo=cta.load_option_daily_total("oi","call","monthly").astype(float)
skew10=cta.load_put_skew(otm_pct=0.10).astype(float); skew5=cta.load_put_skew(otm_pct=0.05).astype(float)
def _tm(p,i,m): return cta.load_three_majors(p,i,m).astype(float)
te_tp=(_tm("TE","投信","net_lots")/_tm("TE","自營商","net_lots").replace(0,np.nan)).replace([np.inf,-np.inf],np.nan)
te_ft=(_tm("TE","外資","net_lots")/_tm("TE","投信","net_lots").replace(0,np.nan)).replace([np.inf,-np.inf],np.nan)
tf_fmt=_tm("TF","外資","oi_net_lots")-_tm("TF","投信","oi_net_lots")
tf_it_oi=_tm("TF","投信","oi_net_lots"); mxf_it_oi=_tm("MXF","投信","oi_net_lots")

def load_taiex_spot():
    root=Path("/Users/hsureggie/coding/Research/QuantResearch/Cache/tw/twse"); rows=[]
    for path in sorted(root.rglob("twse_*.csv")):
        ymd=path.stem[-8:]
        try:
            with open(path,encoding="utf-8-sig") as f:
                for row in csv.reader(f):
                    if row and "發行量加權" in row[0]:
                        rows.append((pd.Timestamp(f"{ymd[:4]}-{ymd[4:6]}-{ymd[6:8]}"), float(row[1].replace(",","").strip()))); break
        except: continue
    return pd.DataFrame(rows,columns=["date","taiex"]).set_index("date").sort_index()["taiex"]
spot=load_taiex_spot().reindex(ASSET.index)
carry=(back_close/close-1); basis=(close/spot-1)
EXEC_LAG=2; PNL_KEEP=(cta.Date("dom")<17)|(cta.Date("dom")>23); SIG_KEEP=PNL_KEEP.shift(-EXEC_LAG).fillna(True).astype(bool)
hlr=(high-low)/close; ev_qc=cta.Event("tsmc_quarterly_call"); ev_exp=cta.Event("futures_expire")

sox_close=cta.load_us_index_tw("^SOX",ASSET.index,"close"); spy_close=cta.load_us_index_tw("SPY",ASSET.index,"close")
_lr=np.log(sox_close/spy_close)
_r_s20=sox_close.pct_change(20); _r_p20=spy_close.pct_change(20)
_d20=_r_p20.where(_r_p20.abs()>0.003,np.nan); _amp20=(np.sign(_r_p20)*(_r_s20/_d20)).clip(-10,10)
_r1s=sox_close.pct_change(1); _r1p=spy_close.pct_change(1)
_rc=_r1s.rolling(60,min_periods=20).corr(_r1p); _p5=sox_close.pct_change(5)*spy_close.pct_change(5)
def _pair_chg_ratio(A,B,N):
    A_chg=A-A.shift(N); B_chg=B-B.shift(N)
    floor=0.0005*B.abs().rolling(120,min_periods=20).mean().shift(1)
    denom=B_chg.where(B_chg.abs()>floor,np.nan)
    return (A_chg/denom).clip(-20,20)
_ratio_60=_pair_chg_ratio(spot,spy_close,60); _prod_st5=sox_close.pct_change(5)*spot.pct_change(5)
_prod_ts60=spot.pct_change(60)*spy_close.pct_change(60)

# c2c pool — 42 base candidates
POOL_C2C = {
    "LT_conv_dev_W10":_dev(tx_conv,10),"LT_conv_dev_W20":_dev(tx_conv,20),
    "LT_conv_signth_W10":_sign_thresh(tx_conv,10),"LT_conv_signth_W60":_sign_thresh(tx_conv,60),
    "LT_top10npct_selftanh_W20":_bd(_selfz(tx_net10,20)),"LT_top10npct_selftanh_W60":_bd(_selfz(tx_net10,60)),
    "LT_top10npct_rankc_W60":_rank_c(tx_net10,60),"LT_top10npct_signth_W60":_sign_thresh(tx_net10,60),
    "LT_logratio_selftanh_W20":_bd(_selfz(tx_log_ratio,20)),"LT_logratio_rankc_W20":_rank_c(tx_log_ratio,20),
    "LT_top5lead_dev_W20":_dev(tx_top5_lead,20),
    "OPT_put_mo_oi_selftanh_W120":_bd(_selfz(put_mo,120)),"OPT_put_mo_oi_selftanh_W60":_bd(_selfz(put_mo,60)),
    "OPT_call_all_oi_selftanh_W20":_bd(_selfz(call_all,20)),"OPT_call_all_oi_signth_W20":_sign_thresh(call_all,20),
    "OPT_call_mo_oi_selfw_W120":_selfz_winsor(call_mo,120),"OPT_skew10_selfz_W60":_selfz(skew10,60),
    "OPT_skew10_selftanh_W60":_bd(_selfz(skew10,60)),"OPT_skew5_rankc_W10":_rank_c(skew5,10),
    "OPT_put_all_oi_signth_W20":_sign_thresh(put_all,20),
    "TM_TE_trustprop_selftanh_W120":_bd(_selfz(te_tp,120)),"TM_TE_trustprop_selftanh_W60":_bd(_selfz(te_tp,60)),
    "TM_TE_ftrust_selfw_W10":_selfz_winsor(te_ft,10),"TM_TF_fmt_robust_W120":_robust_z(tf_fmt,120),
    "TM_TF_ITrust_oi_selfz_W60":_selfz(tf_it_oi,60),"TM_MXF_ITrust_oi_selfw_W10":_selfz_winsor(mxf_it_oi,10),
    "CB_carry_dm240_tanh":cta.Filter(_dm_tanh(carry,240),SIG_KEEP),"CB_carry_dm120_tanh":cta.Filter(_dm_tanh(carry,120),SIG_KEEP),
    "CB_carry_dm60_tanh":cta.Filter(_dm_tanh(carry,60),SIG_KEEP),"CB_carry_chg20_tanh":cta.Filter(_chg_z_tanh(carry,20,60),SIG_KEEP),
    "CB_basis_dm240_tanh":cta.Filter(_dm_tanh(basis,240),SIG_KEEP),"CB_basis_dm120_tanh":cta.Filter(_dm_tanh(basis,120),SIG_KEEP),
    "CB_basis_dm60_tanh":cta.Filter(_dm_tanh(basis,60),SIG_KEEP),"CB_basis_chg5_tanh":cta.Filter(_chg_z_tanh(basis,5,60),SIG_KEEP),
    "CB_basis_chg20_tanh":cta.Filter(_chg_z_tanh(basis,20,60),SIG_KEEP),
    "EVT_F1_range_earn_L5":cta.EventFFill(hlr,(ev_qc==0).astype(float),0,5),
    "US_log_ratio_pct_chg_W120":_lr-_lr.shift(120),
    "US_sox_agree_amp_N20_robustW20":_robust_z(_amp20,20),
    "US_sox_spy_rollcorr_W60_rz_W120":_robust_z(_rc,120),
    "US_sox_spy_ret_product_N5_pctW20":_p5-_p5.shift(20),
    "TV_sox_taiex_ret_product_N5_pctW20":_prod_st5-_prod_st5.shift(20),
    "TV_taiex_over_spy_chg_ratio_N60_pctW120":_ratio_60-_ratio_60.shift(120),
    "TV_taiex_spy_ret_product_N60_pctW120":_prod_ts60-_prod_ts60.shift(120),
}
if volume is not None:
    vz20=(volume-cta.InstMean(20,volume))/cta.InstStdev(20,volume).replace(0,np.nan)
    POOL_C2C["EVT_volz_x_expE6"]=vz20*(ev_exp==+6).astype(float)
    POOL_C2C["EVT_volz_x_expE7"]=vz20*(ev_exp==+7).astype(float)

# Same set of signals also tested with intraday RD_ongap execution (4 top candidates)
POOL_INTRADAY = {
    "OPT_put_mo_oi_selftanh_W60":  POOL_C2C["OPT_put_mo_oi_selftanh_W60"],
    "LT_logratio_selftanh_W20":    POOL_C2C["LT_logratio_selftanh_W20"],
    "LT_top10npct_signth_W60":     POOL_C2C["LT_top10npct_signth_W60"],
    "US_log_ratio_pct_chg_W120":   POOL_C2C["US_log_ratio_pct_chg_W120"],
}

print(f"POOL_C2C:      {len(POOL_C2C)} candidates (all use shift(2) × c2c)")
print(f"POOL_INTRADAY: {len(POOL_INTRADAY)} candidates (shift(1) × RD_ongap — same signals, different execution)")

# Pre-normalize (this is fine — normalization uses trailing stats only)
def _norm(pool):
    return {n: cta.normalize_signal(
                 s.replace([np.inf,-np.inf],np.nan).fillna(0.0),
                 method="tanh", window=252, force=True).rename(n)
            for n, s in pool.items()}
NORM_C2C = _norm(POOL_C2C)
NORM_INTRA = _norm(POOL_INTRADAY)
print(f"normalized")


## 2. Random-hollow framework + IS-only signal evaluation

For each signal + seed:
1. Generate a random 20% OOS mask (contiguous 20-day blocks)
2. IS = 80% of bars, OOS = 20%
3. Compute PnL series (signal.shift × ret − cost) — uses all bars
4. Sign = sign of mean(pnl on IS bars) — from IS only
5. Signed PnL = raw PnL × sign
6. Compute IS stats (SR, β, DD, pos_yr, SR_of_SR) on IS bars only
7. Gate pass/fail based on IS stats only
8. Store the OOS PnL for aggregation later


In [ ]:
def hollow_mask(index, oos_ratio=0.20, block_size=20, seed=0):
    """Generate a random OOS boolean mask over `index`. True = OOS, False = IS.

    Uses contiguous blocks of `block_size` days to preserve autocorrelation.
    Random 80/20 IS/OOS split at the block level.
    """
    rng = np.random.default_rng(seed)
    n = len(index)
    n_blocks = int(np.ceil(n / block_size))
    block_is_oos = rng.random(n_blocks) < oos_ratio
    mask = np.zeros(n, dtype=bool)
    for i, is_oos in enumerate(block_is_oos):
        if is_oos:
            mask[i*block_size : (i+1)*block_size] = True
    return pd.Series(mask, index=index, name="is_oos")

def pnl_series(sig_normalized, ret, cost, shift_days):
    ex = sig_normalized.shift(shift_days)
    return (ex * ret) - ex.fillna(0).diff().abs() * cost

def _sr(x):
    x = x.dropna()
    return float(np.sqrt(252)*x.mean()/x.std()) if x.std() > 0 else np.nan
def _max_dd_days(x):
    dd = (x.cumsum() - x.cumsum().cummax()) < 0
    if not dd.any(): return 0
    return int(dd.astype(int).groupby((dd != dd.shift()).cumsum()).sum().max())
def _stats(pnl):
    a = pnl.dropna()
    if len(a) < 30: return None
    yr = a.groupby(a.index.year).apply(lambda x: _sr(x)).dropna()
    return {"SR":_sr(a), "ann_ret":a.mean()*252, "ann_vol":a.std()*np.sqrt(252),
            "max_dd_d":_max_dd_days(a),
            "pos_yr":(yr>0).mean() if len(yr) else np.nan,
            "worst_yr":yr.min() if len(yr) else np.nan,
            "SR_of_SR":_sr(yr) if yr.std() else np.nan,
            "n_years":yr.size, "n_bars":len(a)}
def _beta(pnl_a, pnl_bh):
    df_ = pd.concat([pnl_a, pnl_bh], axis=1).dropna()
    if len(df_) < 30 or df_.iloc[:,1].std() == 0: return np.nan
    return df_.cov().iloc[0,1] / df_.iloc[:,1].var()

def gate_pass(st, mkt_beta):
    # Same gates as aggregate § 3 (relaxed slightly since IS is only ~80% of history)
    if st is None: return False
    return (st["SR"] > 0.5 and abs(mkt_beta) < 0.15 and
            (st["SR_of_SR"] or 0) > 0.5 and
            (st["pos_yr"] or 0) >= 0.60 and
            (st["worst_yr"] or -99) > -1.5 and
            st["max_dd_d"] < 500 and
            st["n_years"] >= 5)

print("framework helpers loaded")


## 3. Run the IS-selection + OOS-aggregation for a SINGLE seed

Diagnostic single-seed run: show how many signals pass IS gates, what their IS vs OOS SRs are, and the aggregated OOS PnL.


In [ ]:
def run_one_seed(seed, verbose=True):
    # PRE-COMPUTE all PnL series (mask-independent)
    all_pnls = {}
    for n, sig in NORM_C2C.items():
        all_pnls[("C2C", n)] = pnl_series(sig, ret_c2c, cost_c2c, shift_days=2)
    for n, sig in NORM_INTRA.items():
        all_pnls[("INTRA", n)] = pnl_series(sig, ret_ongap, cost_ongap, shift_days=1)

    # For each candidate, generate its own OOS mask, then IS-select + record OOS
    results = []
    OOS_PNLS = {}   # signal_id → OOS-only pnl series (NaN where IS)
    for cand_id, raw_pnl in all_pnls.items():
        mask_seed = seed * 10000 + hash(cand_id) % 10000    # unique mask per (seed, signal)
        m = hollow_mask(CAL, oos_ratio=0.20, block_size=20, seed=mask_seed)
        raw_pnl = raw_pnl.reindex(CAL)

        # Determine sign from IS-only PnL
        is_pnl = raw_pnl.where(~m, np.nan)
        sign = +1 if is_pnl.mean() >= 0 else -1
        signed_pnl = raw_pnl * sign
        signed_is  = signed_pnl.where(~m, np.nan)
        signed_oos = signed_pnl.where(m,  np.nan)

        # IS stats + gate
        is_st = _stats(signed_is)
        mkt_ret_is = ret_c2c.reindex(CAL).where(~m, np.nan)
        is_beta = _beta(signed_is, mkt_ret_is) if is_st else np.nan

        passed = gate_pass(is_st, is_beta) if is_st else False

        # Compute OOS stats for reporting
        oos_st = _stats(signed_oos) if passed else None

        results.append({"cand_id": cand_id, "sign": sign, "passed": passed,
                         "IS_SR": is_st["SR"] if is_st else np.nan,
                         "IS_beta": is_beta,
                         "IS_SRoSR": is_st["SR_of_SR"] if is_st else np.nan,
                         "IS_posyr": is_st["pos_yr"] if is_st else np.nan,
                         "IS_max_dd": is_st["max_dd_d"] if is_st else np.nan,
                         "OOS_SR": oos_st["SR"] if oos_st else np.nan,
                         "OOS_ann_ret": oos_st["ann_ret"] if oos_st else np.nan,
                         "OOS_n_bars": oos_st["n_bars"] if oos_st else np.nan})
        if passed:
            OOS_PNLS[cand_id] = signed_oos

    df = pd.DataFrame(results)

    # Aggregated OOS PnL: at each date D, average across signals where D is OOS
    if OOS_PNLS:
        oos_frame = pd.DataFrame(OOS_PNLS)
        n_contribs = oos_frame.notna().sum(axis=1)
        agg_oos = oos_frame.mean(axis=1)   # mean across non-NaN, per bar
        agg_oos = agg_oos.where(n_contribs > 0, np.nan)
        agg_stats = _stats(agg_oos)
    else:
        agg_oos = pd.Series(dtype=float); agg_stats = None; n_contribs = None

    if verbose:
        n_pass = df["passed"].sum()
        print(f"=== Seed {seed} ===")
        print(f"  candidates: {len(df)}   IS-passers: {n_pass}   "
              f"({n_pass/len(df)*100:.0f}% of pool)")
        if agg_stats:
            print(f"  aggregated OOS SR: {agg_stats['SR']:+.3f}   "
                  f"ann_ret: {agg_stats['ann_ret']*100:+.2f}%   "
                  f"ann_vol: {agg_stats['ann_vol']*100:.2f}%")
            print(f"  aggregated OOS bars: {agg_stats['n_bars']}   "
                  f"max_dd: {agg_stats['max_dd_d']}d")
    return df, OOS_PNLS, agg_oos, n_contribs

# Single-seed diagnostic
np.random.seed(42)
df_one, oos_pnls_one, agg_oos_one, n_contribs_one = run_one_seed(42)

# Show top-20 by IS_SR (from IS-passers)
print("\n=== Single-seed diagnostic (seed=42): top 20 IS-passers ranked by IS_SR ===")
show = ["cand_id","sign","IS_SR","IS_beta","IS_SRoSR","IS_posyr","IS_max_dd",
        "OOS_SR","OOS_ann_ret","OOS_n_bars"]
display(df_one[df_one["passed"]].sort_values("IS_SR", ascending=False).head(20)[show].round(3))


## 4. Single-seed OOS aggregated PnL — visual test


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={"height_ratios":[3,1]})

# Cum PnL of aggregated OOS
ax = axes[0]
agg_cum = agg_oos_one.dropna().cumsum() * 100
ax.plot(agg_cum.index, agg_cum.values, color="#c62828", lw=1.6, label=f"aggregated OOS PnL (single seed=42)")
ax.axhline(0, color="black", lw=0.5, ls="--", alpha=0.5)
ax.set_ylabel("cum OOS PnL (%)")
ax.set_title(f"Single seed OOS test — {agg_cum.notna().sum()} OOS bars aggregated across "
              f"{df_one['passed'].sum()} IS-passer signals")
ax.legend(fontsize=10); ax.grid(alpha=0.3)

# Number of contributing signals per bar
ax = axes[1]
n_contribs_one.plot(ax=ax, color="#546e7a", lw=0.6)
ax.set_ylabel("# signals contributing")
ax.set_title("Number of IS-passer signals with an OOS-bar at each date")
ax.set_xlabel("date"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Also: IS_SR vs OOS_SR scatter — does high IS predict high OOS?
passers = df_one[df_one["passed"]].dropna(subset=["IS_SR","OOS_SR"])
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(passers["IS_SR"], passers["OOS_SR"], s=45, alpha=0.7, color="#1565c0")
ax.axhline(0, color="black", lw=0.4, ls="--", alpha=0.4)
ax.axvline(0, color="black", lw=0.4, ls="--", alpha=0.4)
# Diagonal (perfect correlation)
lim = max(abs(passers[["IS_SR","OOS_SR"]].values).max()*1.05, 2)
ax.plot([-lim,lim],[-lim,lim], color="black", lw=0.5, ls=":", alpha=0.4, label="perfect match")
# Annotate
for _, r in passers.iterrows():
    ax.annotate(r["cand_id"][1][:22], (r["IS_SR"], r["OOS_SR"]), fontsize=6,
                 xytext=(3,2), textcoords="offset points")
corr = passers[["IS_SR","OOS_SR"]].corr().iloc[0,1]
ax.set_xlabel("IS SR"); ax.set_ylabel("OOS SR")
ax.set_title(f"IS-SR vs OOS-SR per IS-passer signal (single seed)  corr = {corr:+.2f}")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\nIS-SR ↔ OOS-SR correlation across {len(passers)} passers: {corr:+.3f}")
print(f"  Positive → high IS-SR predicts high OOS-SR (real edge)")
print(f"  Zero/negative → IS-SR is noise (overfitting)")


## 5. Monte Carlo — repeat over N seeds


In [ ]:
N_SEEDS = 20    # trade-off: 20 seeds is enough to see distribution, ~30-60s
mc_results = []
seed_oos_series = {}
for seed in range(N_SEEDS):
    df_s, _, agg_oos_s, _ = run_one_seed(seed, verbose=False)
    n_pass = df_s["passed"].sum()
    if agg_oos_s.notna().sum() >= 30:
        st = _stats(agg_oos_s)
        mc_results.append({"seed":seed, "n_passers":n_pass,
                            "OOS_SR":st["SR"] if st else np.nan,
                            "OOS_ann_ret":st["ann_ret"]*100 if st else np.nan,
                            "OOS_ann_vol":st["ann_vol"]*100 if st else np.nan,
                            "OOS_max_dd_d":st["max_dd_d"] if st else np.nan,
                            "OOS_pos_yr":(st["pos_yr"] or 0)*100 if st else np.nan})
        seed_oos_series[seed] = agg_oos_s

mc_df = pd.DataFrame(mc_results)
print(f"=== Monte Carlo over {N_SEEDS} seeds ===\n")
display(mc_df.round(3))
print(f"\n=== Distribution of aggregated OOS SR across seeds ===")
print(f"  mean:    {mc_df['OOS_SR'].mean():+.3f}")
print(f"  median:  {mc_df['OOS_SR'].median():+.3f}")
print(f"  std:     {mc_df['OOS_SR'].std():.3f}")
print(f"  min:     {mc_df['OOS_SR'].min():+.3f}")
print(f"  max:     {mc_df['OOS_SR'].max():+.3f}")
print(f"  fraction of seeds with OOS_SR > 0: {(mc_df['OOS_SR']>0).mean()*100:.0f}%")
print(f"  fraction of seeds with OOS_SR > 1: {(mc_df['OOS_SR']>1).mean()*100:.0f}%")

# Histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(mc_df["OOS_SR"], bins=15, color="#c62828", alpha=0.75, edgecolor="white")
axes[0].axvline(0, color="black", lw=1, ls="--", label="SR=0")
axes[0].axvline(mc_df["OOS_SR"].mean(), color="#1565c0", lw=1.5, label=f"mean={mc_df['OOS_SR'].mean():+.2f}")
axes[0].set_xlabel("OOS SR"); axes[0].set_ylabel("# seeds")
axes[0].set_title(f"Distribution of aggregated OOS SR across {N_SEEDS} seeds")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Cum PnL curves for all seeds overlaid
ax = axes[1]
for seed, s in seed_oos_series.items():
    ax.plot(s.dropna().cumsum()*100, color="#546e7a", lw=0.7, alpha=0.4)
# Highlight median-performing seed
median_seed = mc_df.iloc[mc_df["OOS_SR"].argsort().values[len(mc_df)//2]]["seed"]
median_seed = int(median_seed)
ax.plot(seed_oos_series[median_seed].dropna().cumsum()*100,
         color="#c62828", lw=1.6, label=f"median-SR seed ({mc_df.iloc[len(mc_df)//2]['OOS_SR']:+.2f})")
ax.axhline(0, color="black", lw=0.4, ls="--", alpha=0.5)
ax.set_ylabel("cum OOS PnL (%)")
ax.set_title(f"Aggregated OOS cum PnL — {N_SEEDS} seeds overlaid (grey), median highlighted (red)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 6. Interpretation

**If aggregated OOS SR is consistently > 0** across seeds → the selection captures real edge; the aggregate ensemble's SR is not just overfitting.

**If aggregated OOS SR is near 0** (say |mean| < 0.3) or spread wide → the selected signals mostly noise-fit; the SR ~1.5-2.0 seen in earlier notebooks is inflated by selection bias.

**Distribution shape matters more than any single number**:
- Tight distribution around a positive mean → robust genuine edge
- Wide distribution centered near 0 → high sensitivity to which bars leaked into IS, i.e., overfit

**IS-SR ↔ OOS-SR scatter** (§4):
- Strong positive correlation → IS ranking is meaningful
- Zero/negative → IS ranking predicts nothing about OOS

## Caveats

- **Signal COMPUTATION order — this is critical and correct**: rolling windows (e.g. `InstMean(63, x)`, `_selfz(x, 20)`) depend on prior bars in the raw series. So signals and PnL series are computed FIRST on continuous 2013-2026 history, and the OOS mask is applied ONLY AFTER — never before. If we hollowed bars out FIRST and then computed rolling stats, values right after each OOS block would be distorted (rolling mean missing chunks of window). See `run_seed()`: `all_pnls` are pre-computed on full history, then `raw.where(~m, np.nan)` slices IS bars only for stats/sign/gate — the underlying signal never sees the mask.
- **20-day blocks** preserve autocorrelation within blocks but destroy longer trends. Truly bad drawdown regimes lasting >20 days can end up fully in IS or OOS, biasing individual seeds. Consider varying block size.
- **Gates were kept identical to the aggregate's original** — if selection is real, most of the IS-passers should also pass their own OOS gate. If not, gates were tuned to noise.
- **80/20 split** — with ~3300 bars this gives ~660 OOS bars per signal, enough for meaningful SR estimation.
